# Rheology Analysis: Rotational Drag vs Z-Height
Cone-plate geometry — torque per RPM fitted via 4 methods

In [ ]:
import sys
from pathlib import Path

def _find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "viscometry").is_dir():
            return p
    raise RuntimeError("Could not locate repo root (expected src/viscometry)")

PROJECT_ROOT = _find_repo_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

AUTO_RUNS = PROJECT_ROOT / "results" / "auto_runs"
AUTO_RUNS_LEGACY = PROJECT_ROOT / "results" / "Auto-runs"
ARCHIVE = PROJECT_ROOT / "results" / "runs" / "archive"

# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import least_squares
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})
print('Imports OK')

In [ ]:
# ── Cell 2: Load raw data ─────────────────────────────────────────────────────
RAW_PATH = 'timing_v3.csv'   # adjust path if needed

raw_df = pd.read_csv(RAW_PATH)
print(f'Shape: {raw_df.shape}')
print(f'Z-Height range : {raw_df["Z-Height"].min():.4f}  →  {raw_df["Z-Height"].max():.4f}')
print(f'Elapsed_Time_s : {raw_df["Elapsed_Time_s"].min():.2f}  →  {raw_df["Elapsed_Time_s"].max():.2f}')
raw_df.head(3)

In [ ]:
# ── Cell 3: Column catalogue & RPM extraction ─────────────────────────────────
torque_cols = [c for c in raw_df.columns if c not in ('Z-Height', 'Elapsed_Time_s')]

def parse_col(col):
    parts = col.split('_')
    visc  = parts[0]
    rpm   = float(parts[-1])
    return visc, rpm

col_meta = {col: parse_col(col) for col in torque_cols}

print(f'{len(torque_cols)} torque columns found.')
for col in list(col_meta)[:4]:
    visc, rpm = col_meta[col]
    print(f'  {col!r:45s} → visc={visc}, rpm={rpm}')

In [ ]:
# ── Cell 4: Per-height representative torque (top-10% Elapsed_Time_s) ─────────
def top10_mean(group, col):
    thresh = group['Elapsed_Time_s'].quantile(0.90)
    top    = group.loc[group['Elapsed_Time_s'] >= thresh, col]
    return top.mean()

records = []
for height, grp in raw_df.groupby('Z-Height'):
    row = {'Height': height}
    for col in torque_cols:
        row[col] = top10_mean(grp, col)
    records.append(row)

rep_df = pd.DataFrame(records).sort_values('Height').reset_index(drop=True)
print(f'Representative dataframe shape: {rep_df.shape}')
rep_df.head()

In [ ]:
# ── Cell 5: Backup before any trimming ───────────────────────────────────────
backup_df = rep_df.copy()
print('Backup saved (backup_df).')

In [ ]:
# ── Cell 6: Trim thresholds ───────────────────────────────────────────────────
trim_values = {
    '1kcp_1.073_torque_%_rpm_47'       : -66.22,
    '2kcp_3.345_torque_%_rpm_15'       : -66.22,
    '4kcp_6.603_torque_%_rpm_8'        : -66.34,
    '5kcp_5.861_torque_%_rpm_9'        : -66.34,
    '8kcp_8.946_torque_%_rpm_5.6'      : -66.30,
    '10kcp_9.152_torque_%_rpm_5.5'     : -66.32,
    '12.5kcp_14.576_torque_%_rpm_3.5'  : -66.26,
    '15kcp_19.036_torque_%_rpm_2.6'    : -66.32,
    '20kcp_24.396_torque_%_rpm_2.1'    : -66.24,
    '25kcp_22.760_torque_%_rpm_2.2'    : -66.36,
    '30kcp_31.903_torque_%_rpm_1.7'    : -66.26,
    '35kcp_63.253_torque_%_rpm_0.8'    : -66.24,
    '40kcp_62.756_torque_%_rpm_0.8'    : -66.24,
    '45kcp_40.820_torque_%_rpm_1.2'    : -66.34,
    '50kcp_79.653_torque_%_rpm_0.6'    : -66.24,
    '55kcp_48.553_torque_%_rpm_1'      : -66.44,
    '60kcp_68.953_torque_%_rpm_0.8'    : -66.24,
    '70kcp_87.046_torque_%_rpm_0.5'    : -66.26,
    '75kcp_70.730_torque_%_rpm_0.7'    : -66.34,
    '80kcp_103.800_torque_%_rpm_0.5'   : -66.26,
    '90kcp_102.466_torque_%_rpm_0.5'   : -66.24,
    '95kcp_93.400_torque_%_rpm_0.5'    : -66.32,
    '100kcp_124.033_torque_%_rpm_0.4'  : -66.24,
}

trim_top_values = {
    '1kcp_1.073_torque_%_rpm_47'       : None,
    '2kcp_3.345_torque_%_rpm_15'       : None,
    '4kcp_6.603_torque_%_rpm_8'        : -65.92,
    '5kcp_5.861_torque_%_rpm_9'        : -66.00,
    '8kcp_8.946_torque_%_rpm_5.6'      : -65.92,
    '10kcp_9.152_torque_%_rpm_5.5'     : None,
    '12.5kcp_14.576_torque_%_rpm_3.5'  : None,
    '15kcp_19.036_torque_%_rpm_2.6'    : -66.10,
    '20kcp_24.396_torque_%_rpm_2.1'    : None,
    '25kcp_22.760_torque_%_rpm_2.2'    : None,
    '30kcp_31.903_torque_%_rpm_1.7'    : None,
    '35kcp_63.253_torque_%_rpm_0.8'    : None,
    '40kcp_62.756_torque_%_rpm_0.8'    : None,
    '45kcp_40.820_torque_%_rpm_1.2'    : None,
    '50kcp_79.653_torque_%_rpm_0.6'    : None,
    '55kcp_48.553_torque_%_rpm_1'      : -65.94,
    '60kcp_68.953_torque_%_rpm_0.8'    : None,
    '70kcp_87.046_torque_%_rpm_0.5'    : None,
    '75kcp_70.730_torque_%_rpm_0.7'    : None,
    '80kcp_103.800_torque_%_rpm_0.5'   : None,
    '90kcp_102.466_torque_%_rpm_0.5'   : None,
    '95kcp_93.400_torque_%_rpm_0.5'    : None,
    '100kcp_124.033_torque_%_rpm_0.4'  : None,
}
print('Trim thresholds defined.')

In [ ]:
# ── Cell 7: Apply trim (re-runnable) ─────────────────────────────────────────
updated_df = backup_df.copy()

for col in torque_cols:
    bot = trim_values.get(col)
    top = trim_top_values.get(col)
    if bot is not None:
        updated_df.loc[updated_df['Height'] < bot, col] = np.nan
    if top is not None:
        updated_df.loc[updated_df['Height'] > top, col] = np.nan

nan_counts = updated_df[torque_cols].isna().sum()
print('NaN counts per column after trim:')
print(nan_counts[nan_counts > 0].to_string())

In [ ]:
# ── Cell 8: Per-column height normalisation ───────────────────────────────────
norm_df     = updated_df.copy()
height_norm = {}

for col in torque_cols:
    valid_mask = norm_df[col].notna()
    h_zero = norm_df.loc[valid_mask, 'Height'].min() if valid_mask.any() else norm_df['Height'].min()
    height_norm[col] = norm_df['Height'] - h_zero

print('Height normalisation complete (singularity at h=0, below all valid data).')
for col in list(torque_cols)[:3]:
    valid = norm_df[col].notna()
    h = height_norm[col][valid]
    print(f'  {col[:30]:30s}  h_norm in [{h.min():.4f}, {h.max():.4f}]')

In [ ]:
# ── Cell 9: Rotational drag (T / RPM) ────────────────────────────────────────
drag_df = norm_df[['Height']].copy()

for col in torque_cols:
    _, rpm = col_meta[col]
    drag_df[col] = norm_df[col] / rpm

print('Rotational drag computed.')
drag_df.head(3)

In [ ]:
# ── Cell 9b: Nominal viscosity lookup & correlation helper ────────────────────
def label_to_kcp(label):
    return float(label.lower().replace('kcp', ''))

nominal_visc = {col: label_to_kcp(col_meta[col][0]) for col in torque_cols}

def plot_correlation(ax, A_dict, method_name, color):
    """
    Scatter: nominal viscosity (y) vs extracted factor A (x).
    Fits line through origin: visc = k * A
    Shows ideal y=x dashed line for scale reference.
    Returns (k, R²).
    """
    cols_ok = [c for c in torque_cols
               if A_dict.get(c) is not None and np.isfinite(A_dict[c])]
    x_vals  = np.array([A_dict[c]       for c in cols_ok])
    y_vals  = np.array([nominal_visc[c] for c in cols_ok])
    labels  = [col_meta[c][0]           for c in cols_ok]

    # OLS through origin: k = Σ(xy)/Σ(x²)
    k = np.dot(x_vals, y_vals) / np.dot(x_vals, x_vals)

    y_pred = k * x_vals
    ss_res = np.sum((y_vals - y_pred) ** 2)
    ss_tot = np.sum((y_vals - y_vals.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    x_line = np.linspace(0, x_vals.max() * 1.08, 200)

    ax.scatter(x_vals, y_vals, color=color, s=60, zorder=4,
               edgecolors='k', linewidths=0.5)
    ax.plot(x_line, k * x_line, color=color, lw=2,
            label=f'Fit: visc = {k:.4g} · A   (R² = {r2:.4f})')
    ax.plot(x_line, x_line, 'k--', lw=1.3, alpha=0.45, label='Ideal (slope = 1)')

    for xi, yi, lab in zip(x_vals, y_vals, labels):
        ax.annotate(lab, (xi, yi), fontsize=5.5,
                    xytext=(3, 3), textcoords='offset points', alpha=0.85)

    ax.set_xlabel(f'Extracted Factor A  [{method_name}]', fontsize=10)
    ax.set_ylabel('Nominal Viscosity (kcp)', fontsize=10)
    ax.set_title(f'{method_name}\nCorrelation factor  k = {k:.4g}   |   R² = {r2:.4f}',
                 fontsize=10)
    ax.legend(fontsize=8)
    return k, r2

print('Nominal viscosity table and correlation helper defined.')

In [ ]:
# ── Cell 10: Overview plot — raw drag vs Z-Height ─────────────────────────────
colors_all = cm.viridis(np.linspace(0, 1, len(torque_cols)))

fig, ax = plt.subplots(figsize=(12, 6))
for col, color in zip(torque_cols, colors_all):
    visc, rpm = col_meta[col]
    h    = drag_df['Height']
    drag = drag_df[col]
    mask = drag.notna()
    ax.plot(h[mask], drag[mask], lw=1.4, color=color, label=f'{visc} ({rpm} rpm)')
ax.set_xlabel('Z-Height (mm)')
ax.set_ylabel('Rotational Drag  T/RPM')
ax.set_title('Rotational Drag vs Z-Height (all columns, trimmed)')
ax.legend(fontsize=5.5, ncol=4, loc='upper left')
plt.tight_layout()
plt.show()

## Method 1 — Direct Space:  T/RPM = A/h + B  (Huber Robust Fit)

In [ ]:
# ── Cell 11: M1 fit ──────────────────────────────────────────────────────────
def model_direct(x, A, B):
    return A / x + B

def huber_residuals(params, x, y, delta=1.0):
    r = y - model_direct(x, *params)
    return np.where(np.abs(r) <= delta, r,
                    delta * np.sign(r) * np.sqrt(2 * np.abs(r) / delta - 1))

fit1 = {}
for col in torque_cols:
    valid  = drag_df[col].notna()
    h_raw  = height_norm[col][valid].values
    y_raw  = drag_df[col][valid].values
    pos    = h_raw > 0
    h, y   = h_raw[pos], y_raw[pos]
    if len(h) < 3:
        fit1[col] = None; continue
    try:
        p0  = [y.max() * h.min(), y.min()]
        res = least_squares(huber_residuals, p0, args=(h, y),
                            method='trf', loss='huber', f_scale=0.1)
        A, B = res.x
        h_plot = np.linspace(h.min(), h.max(), 300)
        fit1[col] = dict(A=A, B=B, h_fit=h_plot,
                         y_fit=model_direct(h_plot, A, B), h=h, y=y)
    except:
        fit1[col] = None

print('Method 1 fits complete.')

In [ ]:
# ── Cell 12: M1 panel plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(20, 18))
axes = axes.flatten()
for ax, col in zip(axes, torque_cols):
    visc, rpm = col_meta[col]
    f = fit1.get(col)
    if f is None:
        ax.set_title(f'{visc}\n(no fit)'); continue
    ax.scatter(f['h'], f['y'], s=14, color='steelblue', alpha=0.7, zorder=3)
    ax.plot(f['h_fit'], f['y_fit'], 'r-', lw=1.5)
    ax.set_title(f"{visc} | {rpm} rpm\nA={f['A']:.3g}, B={f['B']:.3g}", fontsize=7)
    ax.set_xlabel('h_norm (mm)', fontsize=6)
    ax.set_ylabel('T/RPM', fontsize=6)
for ax in axes[len(torque_cols):]:
    ax.set_visible(False)
fig.suptitle('Method 1 — Direct Space: T/RPM = A/h + B  (Huber robust)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 13: M1 correlation — Nominal Viscosity vs Factor A ──────────────────
fig, ax = plt.subplots(figsize=(8, 6))
A_dict1 = {col: (fit1[col]['A'] if fit1[col] else None) for col in torque_cols}
k1, r2_1 = plot_correlation(ax, A_dict1, 'Method 1 (Direct, Huber)', 'steelblue')
plt.tight_layout()
plt.show()
print(f'M1  →  k = {k1:.4g},  R² = {r2_1:.4f}')

## Method 2 — Reciprocal Space:  T/RPM = A·(1/h) + B  (WLS, w=h²)

In [ ]:
# ── Cell 14: M2 fit ──────────────────────────────────────────────────────────
fit2 = {}
for col in torque_cols:
    valid = drag_df[col].notna()
    h_raw = height_norm[col][valid].values
    y_raw = drag_df[col][valid].values
    pos   = h_raw > 0
    h, y  = h_raw[pos], y_raw[pos]
    if len(h) < 3:
        fit2[col] = None; continue
    X = 1.0 / h
    w = h ** 2
    W   = np.diag(w)
    Xm  = np.column_stack([X, np.ones_like(X)])
    try:
        coeffs = np.linalg.lstsq(Xm.T @ W @ Xm, Xm.T @ W @ y, rcond=None)[0]
        A, B   = coeffs
        X_plot = np.linspace(X.min(), X.max(), 300)
        h_plot = np.linspace(h.min(), h.max(), 300)
        fit2[col] = dict(A=A, B=B, X=X, y=y, h=h,
                         X_fit=X_plot, y_fit=A*X_plot+B,
                         h_fit=h_plot, y_fit_h=A/h_plot+B)
    except:
        fit2[col] = None

print('Method 2 fits complete.')

In [ ]:
# ── Cell 15: M2 panel plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(20, 18))
axes = axes.flatten()
for ax, col in zip(axes, torque_cols):
    visc, rpm = col_meta[col]
    f = fit2.get(col)
    if f is None:
        ax.set_title(f'{visc}\n(no fit)'); continue
    ax.scatter(f['X'], f['y'], s=14, color='darkorange', alpha=0.7, zorder=3)
    ax.plot(f['X_fit'], f['y_fit'], 'b-', lw=1.5)
    ax.set_title(f"{visc} | {rpm} rpm\nA={f['A']:.3g}, B={f['B']:.3g}", fontsize=7)
    ax.set_xlabel('1/h (mm⁻¹)', fontsize=6)
    ax.set_ylabel('T/RPM', fontsize=6)
for ax in axes[len(torque_cols):]:
    ax.set_visible(False)
fig.suptitle('Method 2 — Reciprocal Space: T/RPM = A·(1/h) + B  (WLS)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 16: M2 correlation ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
A_dict2 = {col: (fit2[col]['A'] if fit2[col] else None) for col in torque_cols}
k2, r2_2 = plot_correlation(ax, A_dict2, 'Method 2 (Reciprocal, WLS)', 'darkorange')
plt.tight_layout()
plt.show()
print(f'M2  →  k = {k2:.4g},  R² = {r2_2:.4f}')

## Method 3 — Log-Log Space:  log(T/RPM) = log(A) − m·log(h)  (OLS)

In [ ]:
# ── Cell 17: M3 fit ──────────────────────────────────────────────────────────
fit3 = {}
for col in torque_cols:
    valid = drag_df[col].notna()
    h_raw = height_norm[col][valid].values
    y_raw = drag_df[col][valid].values
    pos   = (h_raw > 0) & (y_raw > 0)
    h, y  = h_raw[pos], y_raw[pos]
    if len(h) < 3:
        fit3[col] = None; continue
    lh, ly = np.log(h), np.log(y)
    slope, intercept, r, p, se = linregress(lh, ly)
    A = np.exp(intercept)
    m = -slope
    h_plot = np.linspace(h.min(), h.max(), 300)
    fit3[col] = dict(A=A, m=m, r2=r**2, lh=lh, ly=ly, h=h, y=y,
                     h_fit=h_plot, y_fit=A * h_plot**(-m))

print('Method 3 fits complete.')
print('Slope m (expect ≈ 1.0 for Newtonian):')
for col in torque_cols:
    f = fit3.get(col)
    if f:
        visc, _ = col_meta[col]
        print(f'  {visc:20s}  m={f["m"]:.3f}  R²={f["r2"]:.4f}')

In [ ]:
# ── Cell 18: M3 panel plots (log-log) ────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(20, 18))
axes = axes.flatten()
for ax, col in zip(axes, torque_cols):
    visc, rpm = col_meta[col]
    f = fit3.get(col)
    if f is None:
        ax.set_title(f'{visc}\n(no fit)'); continue
    ax.scatter(np.log(f['h']), f['ly'], s=14, color='green', alpha=0.7, zorder=3)
    ax.plot(np.log(f['h_fit']), np.log(f['y_fit']), 'm-', lw=1.5)
    ax.set_title(f"{visc} | {rpm} rpm\nm={f['m']:.3f}, R²={f['r2']:.3f}", fontsize=7)
    ax.set_xlabel('log(h)', fontsize=6)
    ax.set_ylabel('log(T/RPM)', fontsize=6)
for ax in axes[len(torque_cols):]:
    ax.set_visible(False)
fig.suptitle('Method 3 — Log-Log: log(T/RPM) = log(A) − m·log(h)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 19: M3 correlation ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
A_dict3 = {col: (fit3[col]['A'] if fit3[col] else None) for col in torque_cols}
k3, r2_3 = plot_correlation(ax, A_dict3, 'Method 3 (Log-Log, OLS)', 'green')
plt.tight_layout()
plt.show()
print(f'M3  →  k = {k3:.4g},  R² = {r2_3:.4f}')

## Method 4 — Dual-Parameter Linearisation:  (T/RPM)·h = A + B·h  (OLS)

In [ ]:
# ── Cell 20: M4 fit ──────────────────────────────────────────────────────────
fit4 = {}
for col in torque_cols:
    valid = drag_df[col].notna()
    h_raw = height_norm[col][valid].values
    y_raw = drag_df[col][valid].values
    pos   = h_raw > 0
    h, y  = h_raw[pos], y_raw[pos]
    if len(h) < 3:
        fit4[col] = None; continue
    z  = y * h
    Xm = np.column_stack([np.ones_like(h), h])
    coeffs, *_ = np.linalg.lstsq(Xm, z, rcond=None)
    A, B = coeffs
    h_plot = np.linspace(h.min(), h.max(), 300)
    fit4[col] = dict(A=A, B=B, h=h, z=z, y=y,
                     h_fit=h_plot, z_fit=A + B*h_plot,
                     y_fit=(A + B*h_plot)/h_plot)

print('Method 4 fits complete.')

In [ ]:
# ── Cell 21: M4 panel plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(20, 18))
axes = axes.flatten()
for ax, col in zip(axes, torque_cols):
    visc, rpm = col_meta[col]
    f = fit4.get(col)
    if f is None:
        ax.set_title(f'{visc}\n(no fit)'); continue
    ax.scatter(f['h'], f['z'], s=14, color='purple', alpha=0.7, zorder=3)
    ax.plot(f['h_fit'], f['z_fit'], 'k-', lw=1.5)
    ax.set_title(f"{visc} | {rpm} rpm\nA={f['A']:.3g}, B={f['B']:.3g}", fontsize=7)
    ax.set_xlabel('h_norm (mm)', fontsize=6)
    ax.set_ylabel('(T/RPM)·h', fontsize=6)
for ax in axes[len(torque_cols):]:
    ax.set_visible(False)
fig.suptitle('Method 4 — Dual-Parameter: (T/RPM)·h = A + B·h', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 22: M4 correlation ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
A_dict4 = {col: (fit4[col]['A'] if fit4[col] else None) for col in torque_cols}
k4, r2_4 = plot_correlation(ax, A_dict4, 'Method 4 (Dual-Param, OLS)', 'purple')
plt.tight_layout()
plt.show()
print(f'M4  →  k = {k4:.4g},  R² = {r2_4:.4f}')

## Summary

In [ ]:
# ── Cell 23: Summary table ────────────────────────────────────────────────────
summary_rows = []
for col in torque_cols:
    visc, rpm = col_meta[col]
    row = dict(Column=col, Visc_Label=visc, RPM=rpm, Nominal_kcp=nominal_visc[col])
    for tag, store in [('M1_A', fit1), ('M2_A', fit2), ('M3_A', fit3), ('M4_A', fit4)]:
        f = store.get(col)
        row[tag] = round(f['A'], 5) if f else np.nan
    f3 = fit3.get(col)
    row['M3_slope_m'] = round(f3['m'], 4) if f3 else np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
pd.set_option('display.float_format', '{:.4g}'.format)
display(summary_df[['Visc_Label','RPM','Nominal_kcp','M1_A','M2_A','M3_A','M4_A','M3_slope_m']])

In [ ]:
# ── Cell 24: Correlation factor comparison ────────────────────────────────────
print('='*55)
print('Correlation factors  (Nominal Viscosity = k × Factor A)')
print('='*55)
for name, k, r2 in [
    ('M1  Direct   / Huber   ', k1, r2_1),
    ('M2  Reciprocal / WLS   ', k2, r2_2),
    ('M3  Log-Log   / OLS    ', k3, r2_3),
    ('M4  Dual-Param / OLS   ', k4, r2_4),
]:
    print(f'  {name}  k = {k:8.4g}   R² = {r2:.4f}')

In [ ]:
# ── Cell 25: 4-panel correlation overview ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
configs = [
    (axes[0,0], {col: (fit1[col]['A'] if fit1[col] else None) for col in torque_cols},
     'Method 1 (Direct, Huber)', 'steelblue'),
    (axes[0,1], {col: (fit2[col]['A'] if fit2[col] else None) for col in torque_cols},
     'Method 2 (Reciprocal, WLS)', 'darkorange'),
    (axes[1,0], {col: (fit3[col]['A'] if fit3[col] else None) for col in torque_cols},
     'Method 3 (Log-Log, OLS)', 'green'),
    (axes[1,1], {col: (fit4[col]['A'] if fit4[col] else None) for col in torque_cols},
     'Method 4 (Dual-Param, OLS)', 'purple'),
]
for ax, A_dict, name, color in configs:
    plot_correlation(ax, A_dict, name, color)

fig.suptitle('All Methods — Nominal Viscosity vs Extracted Factor A', fontsize=13)
plt.tight_layout()
plt.show()